# Árvores Binárias de Busca (BST) — Tutorial

**COMP0497 — Algoritmos e Estruturas de Dados 1**  
Aula 08

## Objetivos

Ao final deste tutorial você será capaz de:

- Implementar a propriedade da BST em código e verificá-la;
- Programar travessias in-order, pre-order e post-order;
- Implementar busca, mínimo, máximo, sucessor, inserção e remoção;
- Visualizar BSTs e comparar a altura em diferentes ordens de inserção;
- Comparar a implementação Python com a referência em Java moderno (`../codigo-java/BST.java`).

> O código de produção da aula está em **Java com generics**. Aqui usamos Python apenas para visualização rápida e exercícios.

## 1. Propriedade da BST

Para todo nó `x`: todas as chaves na subárvore esquerda são `<= x.key` e todas na direita são `>= x.key`. Isso vale **recursivamente** em cada nó.

Vamos definir um nó simples e verificar a propriedade.

In [ ]:
# Exemplo: nó de BST e verificação da propriedade
from dataclasses import dataclass, field
from typing import Optional, Any

@dataclass
class Node:
    key: Any
    value: Any = None
    left:  Optional['Node'] = None
    right: Optional['Node'] = None

def is_bst(x: Optional[Node], lo=float('-inf'), hi=float('inf')) -> bool:
    if x is None:
        return True
    if not (lo <= x.key <= hi):
        return False
    return is_bst(x.left, lo, x.key) and is_bst(x.right, x.key, hi)

# Árvore da Fig. 12.1(a) de Cormen: chaves {2,3,5,5,7,8}
root = Node(5,
    left=Node(3, left=Node(2), right=Node(5)),
    right=Node(7, right=Node(8)))
print('is_bst?', is_bst(root))

In [ ]:
class Node:
    def __init__(self, key):
        self.key = key
        self.left = None
        self.right = None

def is_bst(node, min_val=float('-inf'), max_val=float('inf')):
    if node is None:
        return True
    if not (min_val < node.key < max_val):
        return False
    return is_bst(node.left, min_val, node.key) and is_bst(node.right, node.key, max_val)

ruim = Node(10)
ruim.left = Node(5)
ruim.right = Node(15)
ruim.left.right = Node(12)

assert is_bst(ruim) is False, 'A árvore deveria violar a propriedade'

## 2. Travessias

Três travessias recursivas naturais. A **in-order** imprime as chaves em ordem crescente.

In [ ]:
# Exemplo: travessias em ordem, pre-ordem e pós-ordem
def inorder(x, out):
    if x is None: return
    inorder(x.left, out)
    out.append(x.key)
    inorder(x.right, out)

def preorder(x, out):
    if x is None: return
    out.append(x.key)
    preorder(x.left, out)
    preorder(x.right, out)

def postorder(x, out):
    if x is None: return
    postorder(x.left, out)
    postorder(x.right, out)
    out.append(x.key)

i, p, q = [], [], []
inorder(root, i); preorder(root, p); postorder(root, q)
print('in-order  :', i)   # ordenado
print('pre-order :', p)
print('post-order:', q)

In [ ]:
class Node:
    def __init__(self, key):
        self.key = key
        self.left = None
        self.right = None

def inorder_iter(raiz):
    saida = []
    pilha = []
    atual = raiz

    while atual is not None or pilha:
        while atual is not None:
            pilha.append(atual)
            atual = atual.left

        atual = pilha.pop()
        saida.append(atual.key)
        atual = atual.right

    return saida

root = Node(5)
root.left = Node(3)
root.left.left = Node(2)
root.right = Node(7)
root.right.left = Node(5)
root.right.right = Node(8)

assert inorder_iter(root) == [2, 3, 5, 5, 7, 8], 'Verifique sua implementação'

## 3. BST completa: busca, inserção, mínimo, sucessor

Vamos implementar uma BST genérica em Python, espelhando a `BST.java` da aula.

In [ ]:
# Exemplo: BST completa estilo Sedgewick (inserção recursiva)
class BST:
    def __init__(self):
        self.root = None

    def get(self, key):
        x = self.root
        while x is not None:
            if   key < x.key: x = x.left
            elif key > x.key: x = x.right
            else: return x.value
        return None

    def put(self, key, value=None):
        self.root = self._put(self.root, key, value)

    def _put(self, x, key, value):
        if x is None: return Node(key, value)
        if   key < x.key: x.left  = self._put(x.left,  key, value)
        elif key > x.key: x.right = self._put(x.right, key, value)
        else:             x.value = value
        return x

    def min(self):
        x = self.root
        while x.left is not None: x = x.left
        return x.key

    def keys_inorder(self):
        out = []; inorder(self.root, out); return out

    def height(self):
        def h(x): return -1 if x is None else 1 + max(h(x.left), h(x.right))
        return h(self.root)

arv = BST()
for k in [15, 6, 18, 3, 7, 17, 20, 13, 9]:
    arv.put(k, f'v{k}')
print('in-order :', arv.keys_inorder())
print('min      :', arv.min())
print('altura   :', arv.height())
print('get(13)  :', arv.get(13))
print('get(42)  :', arv.get(42))

In [ ]:
# Visualização: desenhar a BST com matplotlib
import matplotlib.pyplot as plt

def desenhar(no, x=0.0, y=0.0, dx=2.0, ax=None):
    if ax is None:
        fig, ax = plt.subplots(figsize=(7, 4)); ax.set_axis_off()
    if no is None: return ax
    ax.add_patch(plt.Circle((x, y), 0.25, fill=True, color='#cfe2ff', ec='#0d6efd'))
    ax.text(x, y, str(no.key), ha='center', va='center', fontsize=10, fontweight='bold')
    if no.left:
        ax.plot([x, x - dx], [y - 0.25, y - 1 + 0.25], color='#0d6efd')
        desenhar(no.left,  x - dx, y - 1, dx / 1.8, ax)
    if no.right:
        ax.plot([x, x + dx], [y - 0.25, y - 1 + 0.25], color='#0d6efd')
        desenhar(no.right, x + dx, y - 1, dx / 1.8, ax)
    return ax

ax = desenhar(arv.root); ax.set_aspect('equal'); plt.tight_layout(); plt.show()

In [ ]:
class Node:
    def __init__(self, key):
        self.key = key
        self.left = None
        self.right = None

class BST:
    def __init__(self, root=None):
        self.root = root

def sucessor(arvore: BST, chave):
    atual = arvore.root
    sucessor_encontrado = None

    while atual is not None:
        if chave < atual.key:
            sucessor_encontrado = atual
            atual = atual.left
        elif chave > atual.key:
            atual = atual.right
        else:
            if atual.right is not None:
                temp = atual.right
                while temp.left is not None:
                    temp = temp.left
                sucessor_encontrado = temp
            break

    return sucessor_encontrado.key if sucessor_encontrado is not None else None

n15 = Node(15)
n10 = Node(10)
n20 = Node(20)
n8 = Node(8)
n12 = Node(12)
n17 = Node(17)
n25 = Node(25)
n13 = Node(13)

n15.left = n10
n15.right = n20
n10.left = n8
n10.right = n12
n12.right = n13
n20.left = n17
n20.right = n25

arv = BST(n15)

assert sucessor(arv, 13) == 15
assert sucessor(arv, 15) == 17
assert sucessor(arv, 20) is None

## 4. Remoção (Hibbard)

Três casos: 0 filhos, 1 filho, 2 filhos (substitui pelo sucessor).

In [ ]:
# Exemplo: remoção de Hibbard
def _min(x): return x if x.left is None else _min(x.left)

def _delete_min(x):
    if x.left is None: return x.right
    x.left = _delete_min(x.left); return x

def delete(x, key):
    if x is None: return None
    if   key < x.key: x.left  = delete(x.left,  key)
    elif key > x.key: x.right = delete(x.right, key)
    else:
        if x.right is None: return x.left
        if x.left  is None: return x.right
        t = x; x = _min(t.right); x.right = _delete_min(t.right); x.left = t.left
    return x

arv.root = delete(arv.root, 6)   # 6 tem 2 filhos: caso 3
print('após delete(6):', arv.keys_inorder())
ax = desenhar(arv.root); ax.set_aspect('equal'); plt.tight_layout(); plt.show()

In [ ]:
class Node:
    def __init__(self, key):
        self.key = key
        self.left = None
        self.right = None
        self.size = 1

class BST:
    def __init__(self, root=None):
        self.root = root

    def put(self, key):
        self.root = self._put(self.root, key)

    def _put(self, no, key):
        if no is None:
            return Node(key)
        if key < no.key:
            no.left = self._put(no.left, key)
        elif key > no.key:
            no.right = self._put(no.right, key)

        no.size = 1 + size(no.left) + size(no.right)
        return no

def size(no):
    if no is None:
        return 0
    return no.size

n15 = Node(15)
n10 = Node(10)
n20 = Node(20)
n8 = Node(8)
n12 = Node(12)
n17 = Node(17)
n25 = Node(25)
n13 = Node(13)

n15.left = n10
n15.right = n20
n10.left = n8
n10.right = n12
n12.right = n13
n20.left = n17
n20.right = n25

n15.size = 8
n10.size = 4
n20.size = 3
n8.size = 1
n12.size = 2
n13.size = 1
n17.size = 1
n25.size = 1

arv = BST(n15)

assert size(None) == 0
assert size(arv.root) == 8

## 5. Altura: ordem de inserção importa

Vamos comparar a altura de uma BST construída com chaves **aleatórias** vs **ordenadas**.

In [ ]:
# Visualização: altura cresce com n, em log para aleatório, linear para ordenado
import random, math

def altura_para(n, ordem):
    arv = BST()
    for k in ordem: arv.put(k)
    return arv.height()

ns = [10, 50, 100, 200, 500, 1000, 2000]
alts_rand, alts_ord = [], []
for n in ns:
    chaves = list(range(n))
    alts_ord.append(altura_para(n, chaves))
    random.seed(42); random.shuffle(chaves)
    alts_rand.append(altura_para(n, chaves))

plt.figure(figsize=(7, 4))
plt.plot(ns, alts_rand, 'o-', label='ordem aleatória')
plt.plot(ns, alts_ord,  's-', label='ordem crescente (pior caso)')
plt.plot(ns, [2*math.log2(n) for n in ns], 'k--', alpha=0.5, label='2·log₂(n)')
plt.xlabel('n (número de chaves)'); plt.ylabel('altura h')
plt.legend(); plt.title('Altura da BST vs ordem de inserção'); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## Desafio Final

Implemente uma **agenda telefônica** usando a BST genérica `BST.java` (em `../codigo-java/`). A agenda deve:

1. Inserir contatos (`nome -> telefone`), mantendo unicidade do nome;
2. Buscar telefone por nome;
3. Listar contatos em ordem alfabética;
4. Listar contatos cujo nome começa com uma letra dada (use sucessor in-order);
5. Remover um contato.

Adicionalmente, **meça a altura** da BST resultante após inserir 1000 nomes aleatórios e compare com $\lceil 2 \log_2 1000 \rceil \approx 20$. Discuta:

- O que aconteceria se você inserisse os nomes em ordem alfabética?
- Como uma árvore balanceada (próxima aula) resolveria isso?

In [ ]:
class Node:
    def __init__(self, key, value):
        self.key, self.value = key, value
        self.left = self.right = None
        self.size = 1

class BST:
    def __init__(self):
        self.root = None

    def size(self):
        return self.root.size if self.root else 0

    def _size(self, x):
        return x.size if x else 0

    def put(self, key, value):
        self.root = self._put(self.root, key, value)

    def _put(self, x, key, value):
        if not x: return Node(key, value)
        if key < x.key: x.left = self._put(x.left, key, value)
        elif key > x.key: x.right = self._put(x.right, key, value)
        else: x.value = value
        x.size = 1 + self._size(x.left) + self._size(x.right)
        return x

    def get(self, key):
        x = self.root
        while x:
            if key < x.key: x = x.left
            elif key > x.key: x = x.right
            else: return x.value
        return None

    def keys(self):
        res, stack, x = [], [], self.root
        while stack or x:
            while x:
                stack.append(x)
                x = x.left
            x = stack.pop()
            res.append(x.key)
            x = x.right
        return res

    def delete(self, key):
        self.root = self._delete(self.root, key)

    def _delete(self, x, key):
        if not x: return None
        if key < x.key: x.left = self._delete(x.left, key)
        elif key > x.key: x.right = self._delete(x.right, key)
        else:
            if not x.right: return x.left
            if not x.left: return x.right
            t = x
            x = self._min(t.right)
            x.right = self._deleteMin(t.right)
            x.left = t.left
        x.size = 1 + self._size(x.left) + self._size(x.right)
        return x

    def _min(self, x):
        while x.left: x = x.left
        return x

    def _deleteMin(self, x):
        if not x.left: return x.right
        x.left = self._deleteMin(x.left)
        x.size = 1 + self._size(x.left) + self._size(x.right)
        return x

    def ceiling(self, key):
        x, best = self.root, None
        while x:
            if key == x.key: return x.key
            if key < x.key: best, x = x.key, x.left
            else: x = x.right
        return best


class Agenda:
    def __init__(self):
        self._arv = BST()

    def inserir(self, nome: str, telefone: str):
        self._arv.put(nome, telefone)

    def buscar(self, nome: str):
        return self._arv.get(nome)

    def listar_em_ordem(self):
        return self._arv.keys()

    def listar_por_letra(self, letra: str):
        res = []
        letra = letra.upper()
        chave = self._arv.ceiling(letra)
        while chave and chave.upper().startswith(letra):
            res.append(chave)
            chave = self._arv.ceiling(chave + '\0')
        return res

    def remover(self, nome: str):
        self._arv.delete(nome)


a = Agenda()
for n, t in [('Carla', '99'), ('Ana', '11'), ('Bruno', '55'), ('Diego', '77')]:
    a.inserir(n, t)

assert a.listar_em_ordem() == ['Ana', 'Bruno', 'Carla', 'Diego']
assert a.buscar('Bruno') == '55'

## Referências

Veja `../referencias.bib`. Em particular:

- **Cormen et al. (2009)**, *Introduction to Algorithms*, cap. 12 — pseudocódigo e provas de complexidade.
- **Sedgewick (2003)**, *Algorithms in Java, Parts 1–4*, cap. 12 — estilo da implementação recursiva em Java usado em `../codigo-java/BST.java`.